# Проверка гипотезы в Python и составление аналитической записки

## Цели и задачи проекта

Цель проекта - проверить гипотезу: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении Я Книги, чем пользователи из Москвы.

## Описание данных

Таблица bookmate.audition содержит данные об активности пользователей и состоит из следующих полей:  

`audition_id` — уникальный идентификатор сессии чтения или прослушивания;  
`puid` — идентификатор пользователя;  
`usage_platform_ru` — название платформы, с помощью которой пользователь слушал контент;  
`msk_business_dt_str` — дата события в формате строки (московское время);  
`app_version` — версия приложения, которая использовалась для чтения или прослушивания;  
`adult_content_flg` — был ли это контент для взрослых: True или False;  
`hours` — длительность чтения или прослушивания в часах;  
`hours_sessions_long` — продолжительность длинных сессий чтения или прослушивания в часах;  
`kids_content_flg` — был ли это детский контент: True или False;  
`main_content_id` — идентификатор основного контента;  
`usage_geo_id` — идентификатор географического местоположения.  

Таблица `bookmate.content` содержит данные о контенте и состоит из следующих полей:  

`main_content_id` — идентификатор основного контента;  
`main_author_id` — идентификатор основного автора контента;  
`main_content_type` — тип контента;  
`main_content_name` — название контента;  
`main_content_duration_hours` — длительность контента в часах;  
`published_topic_title_list` — список жанров контента.  

Таблица `bookmate.author` содержит данные об авторах контента и состоит из следующих полей:  

`main_author_id` — идентификатор основного автора контента;  
`main_author_name` — имя основного автора контента.  

Таблица `bookmate.geo` содержит данные о местоположении и состоит из следующих полей:  

`usage_geo_id` — идентификатор географического положения;  
`usage_geo_id_name` — город или регион географического положения;  
`usage_country_name` — страна географического положения.  

## Содержимое проекта


Загрузка данных и знакомство с ними  
Проверка гипотезы в Python  
Аналитическая записка  

---

## 1. Загрузка данных и знакомство с ними


In [2]:
# Загружаем библиотеки которые понадобятся для дальнейших расчетов 
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import math
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest

In [3]:
# Загружаем данные из датасета
df = pd.read_csv('/datasets/yandex_knigi_data.csv')

In [4]:
# Оцениваем визуально имеющиеся данные
print("ЗАГРУЗКА ДАННЫХ")
print(f"Размер данных: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())

ЗАГРУЗКА ДАННЫХ
Размер данных: 8784 строк, 4 столбцов


,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


In [5]:
# Смотрим информацию о каждом столбце: тип данных, количество непустых значений
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


In [6]:
# Проверяем дубликаты по puid
duplicates = df['puid'].duplicated().sum()
display(f"Количество дубликатов puid: {duplicates}")

'Количество дубликатов puid: 244'

In [7]:
# Удаляем дубликаты по puid
initial_count = len(df)
df = df.drop_duplicates(subset='puid', keep='first')
final_count = len(df)

display(f"Удалено строк: {initial_count - final_count}")
display(f"Осталось строк: {final_count}")

'Удалено строк: 244'

'Осталось строк: 8540'

In [8]:
# Смотрим размер групп
display(df['city'].value_counts())

# Смотрим статистики по часам для каждого города
display(df.groupby('city')['hours'].describe())

Москва             6234
Санкт-Петербург    2306
Name: city, dtype: int64

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Москва,6234.0,10.881092,36.851683,0.000018,0.059903,0.924498,5.939972,857.209373
Санкт-Петербург,2306.0,11.264433,39.831755,0.000025,0.060173,0.875355,6.138424,978.764775


## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. 

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [9]:
# Разделяем группы по городам
spb = df[df['city'] == 'Санкт-Петербург']['hours']
msk = df[df['city'] == 'Москва']['hours']

# Односторонний t-тест (alternative='greater' означает: среднее СПб > среднего Москвы)
t_stat, p_value = stats.ttest_ind(spb, msk, alternative='greater', equal_var=False)

print(f"t-статистика: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Уровень значимости α = 0.05")

t-статистика: 0.4028
p-value: 0.343571
Уровень значимости α = 0.05


## 3. Аналитическая записка
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:

- Выбранный тип t-теста и уровень статистической значимости. - `Использован односторонний t-тест для независимых выборок (критерий Уэлча). Уровень значимости — 0,05.`

- Результат теста, или p-value. - `В результате проведения t-теста получено значение t-статистики, равное 0,4028. Соответствующее ему одностороннее p-value составляет 0,3436, что превышает установленный порог в 0,05.`

- Вывод на основе полученного p-value, то есть интерпретацию результатов. - `Нулевая гипотеза не отклоняется. Нет статистически значимых доказательств того, что пользователи из Санкт-Петербурга проводят в приложении больше времени, чем пользователи из Москвы. Разница в средних (11,26 ч против 10,88 ч) скорее всего случайна.`

- Одну или две возможные причины, объясняющие полученные результаты. - `Первая причина — очень большой разброс в данных. Есть пользователи с огромным временем активности, которые сильно влияют на среднее. Вторая причина — группы неравные по размеру (Москва почти в 3 раза больше). Это снижает точность теста и мешает заметить разницу, даже если она есть.`



----